# Day 2: working with multidimensional data using Xarray

**Before you start:** make sure all files from data_for_xarra are in the
same folder as this notebook.

A note on scope: this is **quick and exploratory** look at Xarray functionality. We will show how to do basic operations on multidimensional arrays including:
- selecting time steps and time slices
- basic spatial subsetting
- basic calculations - derivation of anomalies
- basic (default) plotting

More advanced processing - such as resampling and regridding (up and down) will be handled in later sessions

## Demonstration - Xarray

### Understanding Xarray objects

In [ ]:
#Import Xarray
import xarray as xr

#importing pyplot for plotting
from matplotlib import pyplot as plt

In [ ]:
#reading some gridded data
ds=xr.open_dataset("./tasmean_mon_ERA5_MDG.nc")

#inspecting its type
type(ds)

In [ ]:
#this is structure of xarray dataset, and it replicates a structure of a netcdf data format
# so there is a:
#    Data variable{s}
#    Dimensions - these are other variables in the file that are not "data" per se
#    Coordinates - this is the place where values for the coordinates are stored
#    Attributes - this is metadata, i.e information about this file and its contents
# dataset can store many different variables. Here we have only pr (and spatial_ref(), which we igrore for the time being)
ds

In [ ]:
# one can access one variable only, e.g. temperature - stored as tasmean
ds["tasmean"]

#this is no longer a Dataset - this is a DataArray. It has a similar structure to a Dataset, 
# but contains only one variable, and coordinates and attributes that are relevant to that variable

In [ ]:
#we can extract that variable from the dataset for further use
# note that DataArray has shape - it is simply a multi-dimensional array of data, but with some additions 
# such as coordinates and date/time
da=ds["tasmean"]
da.shape

In [ ]:
#coordinates of DataArray can be accessed as follows:
da.time

In [ ]:
#this exposes the underlying numpy array
da.data

## Selecting and subsetting data arrays

In [ ]:
#DataArrays can be subset in simlar way to numpy arrays
da[0,:,:]

In [ ]:
#There are several other ways of selecting and subsetting:
#isel selects from dimension by index, not by a value of a coordinate, note that we do not have to worry what is the position of that dimension
da.isel(time=0)

In [ ]:
#we can select by a particular value of the coordinate
# useig a selector .sel() and we have to define which dimension or coordinate we want to select/subset
#for time:
da.sel(time="1994")
#this selects all months for 2024

In [ ]:
#we can select by a particular value of the coordinate
# useig a selector .sel() and we have to define which dimension or coordinate we want to select/subset
#for time:
da.sel(time="1994-01")
#this selects january only

In [ ]:
# selecting one value over latitude:
#note argument method="nearest", if we did not use it - we would need to give exactly the same value as is present in the data
da.sel(lat=-22, method="nearest")

In [ ]:
#selecting a particular grid cell
#note that the object plots in a similar way to Pandas array, but it remains an Xarray DataArray
da.sel(lat=-22, lon=44, method="nearest")

In [ ]:
# to select a range of values - we use slice
# slicing over time:
da.sel(time=slice("2020-Jan", "2020-Jun"))

## Let's plot some data when we are learning about selecting, subsetting
For the time being we will make "default" plots, without customizing them

In [ ]:
# to plot a time series at a location:
da.sel(lat=-22, lon=45, method="nearest").plot()

In [ ]:
# we can select a particular month - here for December only
tasdec=da[da.time.dt.month==12]
tasdec.sel(lat=-22, lon=45, method="nearest").plot()

## aggregating data

In [ ]:
# we can average over the entire domain
# note that two dimensions are in square bracket
da.mean(["lat","lon"]).plot()
# this gives us one dimensional data, i.e. only along time dimension

In [ ]:
#similarly to pandas, we can aggregate data to coarser time steps, e.g. from montly to annual "Y" denotes frequency, 
ydata=da.resample(time="YE").mean()
#for plot, we will do the domain average: 
ydata.mean(["lat","lon"]).plot()


In [ ]:
#plot a map of mean over time
da.mean("time").plot()

In [ ]:
#Hovemuller plot
da.sel(lon=46, method="nearest").plot()

## climatologies and anomalies

In [ ]:
#calculating climatology
clim=da.groupby("time.month").mean("time")
clim

In [ ]:
#calculating anomalies from climatology
#note that we are grouping, but then we do not apply any function on grouped data other than peforming an arightmetic operation. After that - groups "unravel" back to the original shape
# this is unlike in Pandas, where we had to use .transform()
anom=da.groupby("time.month")-clim
#plotting time series of anomalies at a point
anom.sel(lat=-18, lon=47, method="nearest").plot()
plt.show()
#plotting anomaly map for a particular date
anom.sel(time="2000-Jun").plot()